In [1]:
from scipy.stats import ttest_rel
def get_t_test_res(lst1, lst2):
    t_stat, p_value_two_sided = ttest_rel(lst1, lst2)
    
    if t_stat > 0:
        p_value = p_value_two_sided / 2
    else:
        p_value = 1 - (p_value_two_sided / 2)
    
    return t_stat, p_value

def get_t_test_res_rev(lst2, lst1):
    t_stat, p_value_two_sided = ttest_rel(lst1, lst2)
    
    if t_stat > 0:
        p_value = p_value_two_sided / 2
    else:
        p_value = 1 - (p_value_two_sided / 2)
    
    return t_stat, p_value

Our hypotheses:
1. prompt > default over all the languages
2. replace > default over Chinese and Japanese, and replace < default over Arabic, French, Russian
3. prompt > replace over all the languages

check comet statistical results on 6060

In [2]:
import json
import pandas as pd
tgt_langs = [
    "Arabic",
    "Chinese",
    "French",
    "Japanese",
    "Russian",
]
models = ['nllb', 'seamless', 'gpt4omini', 'aya_old', 'aya']
exp_dict = {"default": "_comet", "prompt": "_prompt_gpt4omini_comet", "replace": "_hard_replace_comet", "replace_morphology": "_hard_replace_morphology_comet"}

tot_dfs = {tgt_lang: None for tgt_lang in tgt_langs}
for model in models:
    dataframe_dict = {}
    for exp_name, exp_suffix in exp_dict.items():
        column_name = f"{exp_name}"
        dataframe_dict[column_name] = []
        
        data = json.load(open(f"/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/data_eval_6060/eval/gold_predictions_dev_{model}{exp_suffix}.json"))
        for lang in tgt_langs:
            dataframe_dict[column_name].append(data.get(f"{lang}_scores", []))  # Use `.get()` with default value `0` if lang not found

    # Create the DataFrame
    df = pd.DataFrame(dataframe_dict, index=tgt_langs)
    
    new_dataframes = {}
    for idx, row in df.iterrows():
        # Create a new dataframe where columns are the same, and rows are expanded from the lists
        new_dataframes[idx] = pd.DataFrame({col: row[col] for col in df.columns})
    
    for tgt_lang in tgt_langs:
        if tot_dfs[tgt_lang] is None:
            tot_dfs[tgt_lang] = new_dataframes[tgt_lang]
        else:
            tot_dfs[tgt_lang] = pd.concat([tot_dfs[tgt_lang], new_dataframes[tgt_lang]], ignore_index=True)
            

In [3]:
tot_dfs['Arabic']

,default,prompt,replace,replace_morphology
0,0.790333,0.833181,0.813661,0.842006
1,0.859716,0.859716,0.859716,0.859716
2,0.806007,0.806007,0.806007,0.806007
3,0.903552,0.903552,0.903552,0.903552
4,0.937370,0.937370,0.938195,0.937370
...,...,...,...,...
2335,0.777188,0.777240,0.766903,0.783127
2336,0.884026,0.888678,0.768463,0.888677
2337,0.653796,0.802058,0.643497,0.855060
2338,0.807697,0.852741,0.857072,0.857072


In [8]:
dfs = pd.DataFrame()
for tgt_lang in tgt_langs:
    info = []
    # test hypothesis 1
    t_stat, p_val = get_t_test_res(tot_dfs[tgt_lang]['prompt'].tolist(), tot_dfs[tgt_lang]['default'].tolist())
    info.append({"hypothesis": 1, "t_stat": t_stat, "p_val": p_val})
    
    # test hypothesis 2
    t_stat, p_val = get_t_test_res(tot_dfs[tgt_lang]['replace'].tolist(), tot_dfs[tgt_lang]['default'].tolist()) if tgt_lang in ['Chinese', 'Japanese'] else get_t_test_res(tot_dfs[tgt_lang]['default'].tolist(), tot_dfs[tgt_lang]['replace'].tolist())
    info.append({"hypothesis": 2, "t_stat": t_stat, "p_val": p_val})
    
    # test hypothesis 3
    t_stat, p_val = get_t_test_res(tot_dfs[tgt_lang]['prompt'].tolist(), tot_dfs[tgt_lang]['replace'].tolist())
    info.append({"hypothesis": 3, "t_stat": t_stat, "p_val": p_val})
    
    df = pd.DataFrame(info)
    if dfs.shape[0] == 0:
        dfs = df.rename(columns={'t_stat': f"t_stats_{tgt_lang}", 'p_val': f"p_val_{tgt_lang}"})
    else:
        dfs[f"t_stats_{tgt_lang}"] = df['t_stat']
        dfs[f"p_val_{tgt_lang}"] = df['p_val']
    

In [ ]:
dfs = dfs.set_index('hypothesis')
# print(dfs.to_markdown(floatfmt=".2f"))
print(dfs.to_latex(float_format=lambda x: f"{x:.2f}"))

\begin{tabular}{lrrrrrrrrrrr}
\toprule
 & hypothesis & t_stats_Arabic & p_val_Arabic & t_stats_Chinese & p_val_Chinese & t_stats_French & p_val_French & t_stats_Japanese & p_val_Japanese & t_stats_Russian & p_val_Russian \\
\midrule
0 & 1 & 14.18 & 0.00 & 15.22 & 0.00 & 12.68 & 0.00 & 11.13 & 0.00 & 8.53 & 0.00 \\
1 & 2 & 6.91 & 0.00 & 7.60 & 0.00 & 1.84 & 0.03 & 3.21 & 0.00 & 20.76 & 0.00 \\
2 & 3 & 17.93 & 0.00 & 11.67 & 0.00 & 13.58 & 0.00 & 10.62 & 0.00 & 24.89 & 0.00 \\
\bottomrule
\end{tabular}



check bleu, chrf, chrf++, ter results on 6060

In [11]:
import pandas as pd
tgt_langs = [
    "Arabic",
    "Chinese",
    "French",
    "Japanese",
    "Russian",
]
models = ['nllb', 'seamless', 'gpt4omini', 'aya_old', 'aya']
exp_dict = {"default": "", "prompt": "_prompt_gpt4omini", "replace": "_hard_replace", "replace_morphology": "_hard_replace_morphology"}

for metric in ["_bleu_score", "_chrf_score", "_chrfpp_score", "_ter_score"]:
    tot_dfs = {tgt_lang: None for tgt_lang in tgt_langs}
    for model in models:
        dataframe_dict = {}
        for exp_name, exp_suffix in exp_dict.items():
            column_name = f"{exp_name}"
            dataframe_dict[column_name] = []

            data = pd.read_csv(f"/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/data_eval_6060/eval/gold_predictions_dev_{model}{exp_suffix}.csv")
            for lang in tgt_langs:
                dataframe_dict[column_name].append(data[f"{lang}{metric}"].tolist())

        df = pd.DataFrame(dataframe_dict, index=tgt_langs)

        new_dataframes = {}
        for idx, row in df.iterrows():
            # Create a new dataframe where columns are the same, and rows are expanded from the lists
            new_dataframes[idx] = pd.DataFrame({col: row[col] for col in df.columns})

        for tgt_lang in tgt_langs:
            if tot_dfs[tgt_lang] is None:
                tot_dfs[tgt_lang] = new_dataframes[tgt_lang]
            else:
                tot_dfs[tgt_lang] = pd.concat([tot_dfs[tgt_lang], new_dataframes[tgt_lang]], ignore_index=True)

    dfs = pd.DataFrame()
    for tgt_lang in tgt_langs:
        info = []
        if metric not in ['_ter_score']:
            # test hypothesis 1
            t_stat, p_val = get_t_test_res(tot_dfs[tgt_lang]['prompt'].tolist(), tot_dfs[tgt_lang]['default'].tolist())
            info.append({"hypothesis": 1, "t_stat": t_stat, "p_val": p_val})

            # test hypothesis 2
            t_stat, p_val = get_t_test_res(tot_dfs[tgt_lang]['replace'].tolist(), tot_dfs[tgt_lang]['default'].tolist()) if tgt_lang in ['Chinese', 'Japanese'] else get_t_test_res(tot_dfs[tgt_lang]['default'].tolist(), tot_dfs[tgt_lang]['replace'].tolist())
            info.append({"hypothesis": 2, "t_stat": t_stat, "p_val": p_val})

            # test hypothesis 3
            t_stat, p_val = get_t_test_res(tot_dfs[tgt_lang]['prompt'].tolist(), tot_dfs[tgt_lang]['replace'].tolist())
            info.append({"hypothesis": 3, "t_stat": t_stat, "p_val": p_val})
        else:
            # test hypothesis 1
            t_stat, p_val = get_t_test_res_rev(tot_dfs[tgt_lang]['prompt'].tolist(), tot_dfs[tgt_lang]['default'].tolist())
            info.append({"hypothesis": 1, "t_stat": t_stat, "p_val": p_val})

            # test hypothesis 2
            t_stat, p_val = get_t_test_res_rev(tot_dfs[tgt_lang]['replace'].tolist(), tot_dfs[tgt_lang]['default'].tolist()) if tgt_lang in ['Chinese', 'Japanese'] else get_t_test_res_rev(tot_dfs[tgt_lang]['default'].tolist(), tot_dfs[tgt_lang]['replace'].tolist())
            info.append({"hypothesis": 2, "t_stat": t_stat, "p_val": p_val})

            # test hypothesis 3
            t_stat, p_val = get_t_test_res_rev(tot_dfs[tgt_lang]['prompt'].tolist(), tot_dfs[tgt_lang]['replace'].tolist())
            info.append({"hypothesis": 3, "t_stat": t_stat, "p_val": p_val})

        df = pd.DataFrame(info)
        if dfs.shape[0] == 0:
            dfs = df.rename(columns={'t_stat': f"t_stats_{tgt_lang}", 'p_val': f"p_val_{tgt_lang}"})
        else:
            dfs[f"t_stats_{tgt_lang}"] = df['t_stat']
            dfs[f"p_val_{tgt_lang}"] = df['p_val']            

    dfs = dfs.set_index('hypothesis')
    print(metric)
    # print(dfs.to_markdown(floatfmt=".2f"))
    print(dfs.to_latex(float_format=lambda x: f"{x:.2f}"))

_bleu_score
\begin{tabular}{lrrrrrrrrrr}
\toprule
 & t_stats_Arabic & p_val_Arabic & t_stats_Chinese & p_val_Chinese & t_stats_French & p_val_French & t_stats_Japanese & p_val_Japanese & t_stats_Russian & p_val_Russian \\
hypothesis &  &  &  &  &  &  &  &  &  &  \\
\midrule
1 & 6.37 & 0.00 & 6.72 & 0.00 & 9.80 & 0.00 & 6.26 & 0.00 & 7.63 & 0.00 \\
2 & -0.66 & 0.75 & 3.89 & 0.00 & -0.22 & 0.59 & 2.58 & 0.00 & -2.85 & 1.00 \\
3 & 4.80 & 0.00 & 3.87 & 0.00 & 8.77 & 0.00 & 4.57 & 0.00 & 5.50 & 0.00 \\
\bottomrule
\end{tabular}

_chrf_score
\begin{tabular}{lrrrrrrrrrr}
\toprule
 & t_stats_Arabic & p_val_Arabic & t_stats_Chinese & p_val_Chinese & t_stats_French & p_val_French & t_stats_Japanese & p_val_Japanese & t_stats_Russian & p_val_Russian \\
hypothesis &  &  &  &  &  &  &  &  &  &  \\
\midrule
1 & 6.37 & 0.00 & 6.72 & 0.00 & 9.79 & 0.00 & 6.26 & 0.00 & 7.64 & 0.00 \\
2 & -0.66 & 0.75 & 3.89 & 0.00 & -0.18 & 0.57 & 2.59 & 0.00 & -2.85 & 1.00 \\
3 & 4.79 & 0.00 & 3.86 & 0.00 & 8.76 & 0.0